[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SubSurfObs/observatory_notebooks/blob/main/01_fdsn_access/index.ipynb)

# 01 — Multi-Network FDSN Data Access

This notebook demonstrates how to query station metadata and download seismic waveforms
from multiple FDSN providers using ObsPy.

**Worked example:** Moe, Victoria sequence, 3 February 2026.

**Output:** waveform and inventory files saved to `data/` for use by subsequent notebooks.


<table style="border:1px solid #BFC3D1;border-radius:6px;padding:0.5rem 1rem;margin-bottom:1rem;font-size:0.9em;background:#f9f9fb;border-collapse:separate;">
<tr><td style="padding:2px 12px 2px 0"><strong>Authors</strong></td><td>Dan Sandiford&nbsp;<a href="https://orcid.org/0000-0002-2207-6837"><img src="https://orcid.org/sites/default/files/images/orcid_16x16.png" alt="ORCID" style="vertical-align:middle"> 0000-0002-2207-6837</a></td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Institution</strong></td><td>University of Melbourne — Subsurface Observatory</td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Funding</strong></td><td>AuScope / National Collaborative Research Infrastructure Strategy (NCRIS)</td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Data</strong></td><td>VW network &middot; DOI <a href="https://doi.org/10.7914/8csc-8z27">10.7914/8csc-8z27</a></td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Notebook DOI</strong></td><td><em>in preparation (Zenodo)</em></td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Licence</strong></td><td><a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a></td></tr>
</table>

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'obspy', 'folium', 'pandas'], check=True)
    subprocess.run(['wget', '-q',
        'https://raw.githubusercontent.com/SubSurfObs/observatory_notebooks/main/utils.py'],
        check=True)


In [ ]:
from pathlib import Path
import sys

for p in [Path.cwd().parent, Path.cwd()]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import folium
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.clients.fdsn.header import FDSNNoDataException
from obspy.core.inventory import Inventory
import matplotlib.pyplot as plt

from utils import FDSN_UOM, FDSN_AUSPASS, FDSN_RS
print('Imports OK')


In [ ]:
# Search parameters — Moe, Victoria sequence, 3 Feb 2026
T_START    = UTCDateTime('2026-02-03T06:00:00')
DURATION   = 10 * 60   # seconds
T_END      = T_START + DURATION

SEARCH_LAT = -38.3
SEARCH_LON =  146.6
MAX_RADIUS =  1.5   # degrees

print(f'Window: {T_START}  ->  {T_END}  ({DURATION} s)')


## Station inventory

Each provider is queried independently. We ask each one only for the networks
it is known to serve — UoM for VW, AusPass for OZ, RaspberryShake for AM.
Inventories are merged at the end.


In [ ]:
# UoM FDSNWS — VW network (University of Melbourne)
client_uom = Client(FDSN_UOM)
inv_vw = client_uom.get_stations(
    network='VW',
    latitude=SEARCH_LAT, longitude=SEARCH_LON, maxradius=MAX_RADIUS,
    starttime=T_START, endtime=T_END,
    level='channel',
)
print(f'VW stations: {[s.code for n in inv_vw for s in n]}')


In [ ]:
# AusPass — OZ network (Seismology Research Centre)
client_auspass = Client(FDSN_AUSPASS)
inv_oz = client_auspass.get_stations(
    network='OZ',
    latitude=SEARCH_LAT, longitude=SEARCH_LON, maxradius=MAX_RADIUS,
    starttime=T_START, endtime=T_END,
    level='channel',
)
print(f'OZ stations: {[s.code for n in inv_oz for s in n]}')


In [ ]:
# RaspberryShake — AM network (citizen seismograph network)
client_rs = Client(FDSN_RS)
try:
    inv_am = client_rs.get_stations(
        network='AM',
        latitude=SEARCH_LAT, longitude=SEARCH_LON, maxradius=MAX_RADIUS,
        starttime=T_START, endtime=T_END,
        level='channel',
    )
    print(f'AM stations: {[s.code for n in inv_am for s in n]}')
except FDSNNoDataException:
    inv_am = Inventory()
    print('AM: no stations in this region')


In [ ]:
inv_all = inv_vw + inv_oz + inv_am
print(f'Total stations: {len(list(inv_all.get_contents()["stations"]))}'
      f'  across {len(list(inv_all.get_contents()["networks"]))} networks')


In [ ]:
# Interactive station map
lons, lats, labels, nets = [], [], [], []
for net in inv_all:
    for sta in net:
        if sta.longitude is not None and sta.latitude is not None:
            lons.append(sta.longitude)
            lats.append(sta.latitude)
            labels.append(sta.code)
            nets.append(net.code)

m = folium.Map(
    location=[SEARCH_LAT, SEARCH_LON], zoom_start=8, tiles='OpenStreetMap'
)
NET_COLOUR = {'VW': 'darkblue', 'OZ': 'cadetblue', 'AM': 'purple'}
for lon, lat, label, net in zip(lons, lats, labels, nets):
    folium.Marker(
        location=[lat, lon],
        tooltip=f'{net}.{label}',
        icon=folium.Icon(color=NET_COLOUR.get(net, 'gray'), icon='signal'),
    ).add_to(m)
folium.Marker(
    location=[SEARCH_LAT, SEARCH_LON],
    tooltip='Search centre',
    icon=folium.Icon(color='red', icon='asterisk'),
).add_to(m)
m


## Waveforms

Each provider is queried for waveforms from the stations it returned above.
We fetch all available broadband and strong-motion channels, then merge.


In [ ]:
# VW waveforms from UoM
vw_stations = ','.join(s.code for n in inv_vw for s in n)
st_vw = client_uom.get_waveforms(
    network='VW', station=vw_stations, location='*', channel='*H*,*N*',
    starttime=T_START, endtime=T_END, attach_response=True,
)
print(f'VW: {len(st_vw)} traces')


In [ ]:
# OZ waveforms from AusPass
oz_stations = ','.join(s.code for n in inv_oz for s in n)
st_oz = client_auspass.get_waveforms(
    network='OZ', station=oz_stations, location='*', channel='*H*,*N*',
    starttime=T_START, endtime=T_END, attach_response=True,
)
print(f'OZ: {len(st_oz)} traces')


In [ ]:
# AM waveforms from RaspberryShake (skip if no stations found above)
st_am = None
if len(inv_am) > 0:
    try:
        am_stations = ','.join(s.code for n in inv_am for s in n)
        st_am = client_rs.get_waveforms(
            network='AM', station=am_stations, location='*', channel='*H*,*N*',
            starttime=T_START, endtime=T_END, attach_response=True,
        )
        print(f'AM: {len(st_am)} traces')
    except FDSNNoDataException:
        print('AM: no waveforms returned')
        st_am = None


In [ ]:
from obspy import Stream
st = st_vw + st_oz + (st_am if st_am else Stream())
st.merge(method=1, fill_value='interpolate')
print(f'Total: {len(st)} traces')
print(st)


In [ ]:
st_Z = st.select(channel='*HZ').copy()
st_Z.detrend('demean')
st_Z.filter('bandpass', freqmin=1.0, freqmax=20.0, corners=4)
fig = st_Z.plot(equal_scale=False, size=(900, 600), handle=True)
fig.suptitle('*HZ — bandpass 1-20 Hz', fontsize=11)
plt.tight_layout()
plt.show()


## Save data

Downsample channels above 100 Hz before saving — PhaseNet operates at 100 Hz
so there is no benefit in storing higher-rate data for the downstream notebooks.


In [ ]:
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)

st_save = st.copy()
for tr in st_save:
    if tr.stats.sampling_rate > 100:
        tr.resample(100.0)

mseed_path = data_dir / 'ex01.mseed'
xml_path   = data_dir / 'ex01.xml'

st_save.write(str(mseed_path), format='MSEED')
inv_all.write(str(xml_path),   format='STATIONXML')

print(f'Traces saved: {len(st_save)}')
print(f'{mseed_path}  ({mseed_path.stat().st_size / 1024**2:.1f} MB)')
print(f'{xml_path}    ({xml_path.stat().st_size / 1024:.0f} kB)')


---

**Next:** [02 — Phase Picking with SeisBench](../02_phase_picking/index.ipynb)

**Data source:** University of Melbourne Seismic Network (VW), DOI [10.7914/8csc-8z27](https://doi.org/10.7914/8csc-8z27).  
**Licence:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
